# Prédiction du prix des voitures d'occasion
Dataset fusionné (CarDekho) : `cars_merged_extended.csv` — régression linéaire et Lasso.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn import metrics

In [ ]:
car_dataset = pd.read_csv('/content/cars_merged_extended.csv')

In [ ]:
car_dataset.head()

In [ ]:
car_dataset.shape

In [ ]:
car_dataset.info()

In [ ]:
car_dataset.isnull().sum()

In [ ]:
# checking the distribution of categorical data
print(car_dataset.fuel.value_counts())
print(car_dataset.seller_type.value_counts())
print(car_dataset.transmission.value_counts())
print(car_dataset.brand.value_counts())

In [ ]:
# Opt-in to the future behavior to silence downcasting warnings
pd.set_option('future.no_silent_downcasting', True)

# encoding "fuel" Column
car_dataset.replace({'fuel':{'Petrol':0,'Diesel':1,'CNG':2,'LPG':3}},inplace=True)

# encoding "seller_type" Column
car_dataset.replace({'seller_type':{'Dealer':0,'Individual':1}},inplace=True)

# encoding "transmission" Column
car_dataset.replace({'transmission':{'Manual':0,'Automatic':1}},inplace=True)

# remplace "year" par l'âge de la voiture
car_dataset['age'] = 2022 - car_dataset['year']

# encoding "brand" Column (one-hot)
car_dataset = pd.get_dummies(car_dataset, columns=['brand'], drop_first=True, dtype=int)

In [ ]:
car_dataset.head()

In [ ]:
#Splitting the data and Target
# (selling_price est en lakhs ; on prédit son logarithme, puis on revient en lakhs avec np.exp)
X = car_dataset.drop(['name','year','source','selling_price'],axis=1)
Y = np.log(car_dataset['selling_price'])

In [ ]:
print(X)

In [ ]:
print(Y)

In [ ]:
#Splitting Training and Test data
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, random_state=2)

# standardisation (ajustée sur le train uniquement)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# loading the linear regression model
lin_reg_model = LinearRegression()

In [ ]:
lin_reg_model.fit(X_train,Y_train)

In [ ]:
# prediction on Training data
training_data_prediction = lin_reg_model.predict(X_train)

In [ ]:
# R squared Error
error_score = metrics.r2_score(Y_train, training_data_prediction)
print("R squared Error (log) : ", error_score)
print("R squared Error (lakhs) : ", metrics.r2_score(np.exp(Y_train), np.exp(training_data_prediction)))
print("MAE (lakhs) : ", metrics.mean_absolute_error(np.exp(Y_train), np.exp(training_data_prediction)))

In [ ]:
plt.scatter(np.exp(Y_train), np.exp(training_data_prediction), s=5)
plt.xlabel("Actual Price (lakhs)")
plt.ylabel("Predicted Price (lakhs)")
plt.title(" Actual Prices vs Predicted Prices")
plt.show()

In [ ]:
# prediction on test data
test_data_prediction = lin_reg_model.predict(X_test)

In [ ]:
# R squared Error
error_score = metrics.r2_score(Y_test, test_data_prediction)
print("R squared Error (log) : ", error_score)
print("R squared Error (lakhs) : ", metrics.r2_score(np.exp(Y_test), np.exp(test_data_prediction)))
print("MAE (lakhs) : ", metrics.mean_absolute_error(np.exp(Y_test), np.exp(test_data_prediction)))

In [ ]:
plt.scatter(np.exp(Y_test), np.exp(test_data_prediction), s=5)
plt.xlabel("Actual Price (lakhs)")
plt.ylabel("Predicted Price (lakhs)")
plt.title(" Actual Prices vs Predicted Prices")
plt.show()

In [ ]:
#lasso_regression

In [ ]:
# loading the lasso regression model
# LassoCV choisit automatiquement le meilleur alpha par validation croisée
# (Lasso() avec alpha=1 annulerait presque tous les coefficients sur ces données)
lass_reg_model = LassoCV(cv=5)

In [ ]:
lass_reg_model.fit(X_train,Y_train)

In [ ]:
#model evaluation
# prediction on Training data
training_data_prediction = lass_reg_model.predict(X_train)

In [ ]:
# R squared Error
error_score = metrics.r2_score(Y_train, training_data_prediction)
print("R squared Error (log) : ", error_score)
print("R squared Error (lakhs) : ", metrics.r2_score(np.exp(Y_train), np.exp(training_data_prediction)))
print("MAE (lakhs) : ", metrics.mean_absolute_error(np.exp(Y_train), np.exp(training_data_prediction)))
print("alpha choisi : ", lass_reg_model.alpha_)

In [ ]:
#visualize the result
plt.scatter(np.exp(Y_train), np.exp(training_data_prediction), s=5)
plt.xlabel("Actual Price (lakhs)")
plt.ylabel("Predicted Price (lakhs)")
plt.title(" Actual Prices vs Predicted Prices")
plt.show()

In [ ]:
# prediction on Test data
test_data_prediction = lass_reg_model.predict(X_test)

In [ ]:
# R squared Error
error_score = metrics.r2_score(Y_test, test_data_prediction)
print("R squared Error (log) : ", error_score)
print("R squared Error (lakhs) : ", metrics.r2_score(np.exp(Y_test), np.exp(test_data_prediction)))
print("MAE (lakhs) : ", metrics.mean_absolute_error(np.exp(Y_test), np.exp(test_data_prediction)))

In [ ]:
plt.scatter(np.exp(Y_test), np.exp(test_data_prediction), s=5)
plt.xlabel("Actual Price (lakhs)")
plt.ylabel("Predicted Price (lakhs)")
plt.title(" Actual Prices vs Predicted Prices")
plt.show()